# 03 – Player Analysis

Deep-dive into individual player performance across seasons.

**Analyses included:**
- Top scorers, rebounders, and assist leaders
- Season scoring trends for elite players
- Correlation heatmap of per-game stats
- K-Means clustering to identify player archetypes

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid')
%matplotlib inline

RAW_DIR = os.path.join(os.path.abspath('..'), 'data', 'raw')
PROCESSED_DIR = os.path.join(os.path.abspath('..'), 'data', 'processed')

## 1. Load Player Season Stats

In [ ]:
# Load pre-built season stats (from notebook 02) or build on the fly
season_path = os.path.join(PROCESSED_DIR, 'player_season_stats.csv')
if os.path.exists(season_path):
    player_season = pd.read_csv(season_path)
else:
    from src.data_loader import load_games, load_games_details
    from src.features import add_true_shooting, add_usage_rate, aggregate_player_season_stats
    games = load_games(RAW_DIR)
    details = load_games_details(RAW_DIR)
    if 'SEASON' in games.columns:
        season_map = games.set_index('GAME_ID')['SEASON']
        details['SEASON'] = details['GAME_ID'].map(season_map)
    details = add_true_shooting(details)
    details = add_usage_rate(details)
    player_season = aggregate_player_season_stats(details)

# Merge player names if available
from src.data_loader import load_players
players = load_players(RAW_DIR)
name_col = next((c for c in players.columns if 'NAME' in c.upper()), None)
id_col = next((c for c in players.columns if 'PLAYER_ID' in c.upper()), None)
if name_col and id_col:
    name_map = players.set_index(id_col)[name_col]
    player_season['PLAYER_NAME'] = player_season['PLAYER_ID'].map(name_map).fillna('Unknown')

print(player_season.shape)
player_season.head()

## 2. Top Scorers (career points per game, min 50 games)

In [ ]:
career = player_season.groupby('PLAYER_ID').agg(
    GP=('GP', 'sum'),
    PTS_PG=('PTS_PG', 'mean'),
    REB_PG=('REB_PG', 'mean'),
    AST_PG=('AST_PG', 'mean'),
    PLAYER_NAME=('PLAYER_NAME', 'first') if 'PLAYER_NAME' in player_season.columns else ('PLAYER_ID', 'first'),
).reset_index()

qualified = career[career['GP'] >= 50]
top_scorers = qualified.nlargest(15, 'PTS_PG')

plt.figure(figsize=(11, 5))
bars = plt.barh(top_scorers['PLAYER_NAME'], top_scorers['PTS_PG'], color='royalblue')
plt.xlabel('Points Per Game')
plt.title('Top 15 Scorers (≥50 games, career average)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Top Rebounders & Assisters

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_reb = qualified.nlargest(15, 'REB_PG')
axes[0].barh(top_reb['PLAYER_NAME'], top_reb['REB_PG'], color='seagreen')
axes[0].set_xlabel('Rebounds Per Game')
axes[0].set_title('Top 15 Rebounders')
axes[0].invert_yaxis()

top_ast = qualified.nlargest(15, 'AST_PG')
axes[1].barh(top_ast['PLAYER_NAME'], top_ast['AST_PG'], color='darkorange')
axes[1].set_xlabel('Assists Per Game')
axes[1].set_title('Top 15 Assist Leaders')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 4. Correlation Heatmap

In [ ]:
stat_cols = [c for c in ['PTS_PG', 'REB_PG', 'AST_PG', 'STL_PG', 'BLK_PG', 'MIN_PG'] if c in player_season.columns]
corr = player_season[stat_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Heatmap — Per-Game Stats')
plt.tight_layout()
plt.show()

## 5. Player Clustering with K-Means

In [ ]:
cluster_cols = [c for c in ['PTS_PG', 'REB_PG', 'AST_PG', 'STL_PG', 'BLK_PG', 'MIN_PG'] if c in career.columns]
cluster_data = career[cluster_cols].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_data)

# Elbow method
inertias = []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(list(K_range), inertias, 'bo-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.tight_layout()
plt.show()

In [ ]:
# Fit with k=4 player archetypes
K = 4
km = KMeans(n_clusters=K, random_state=42, n_init=10)
career.loc[cluster_data.index, 'CLUSTER'] = km.fit_predict(X_scaled)

# PCA for 2-D visualisation
pca = PCA(n_components=2)
coords = pca.fit_transform(X_scaled)
labels = km.labels_

plt.figure(figsize=(9, 6))
scatter = plt.scatter(coords[:, 0], coords[:, 1], c=labels, cmap='tab10', alpha=0.6, s=40)
plt.colorbar(scatter, label='Cluster')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('Player Archetypes via K-Means Clustering (PCA projection)')
plt.tight_layout()
plt.show()

In [ ]:
# Cluster profiles
cluster_profiles = career.groupby('CLUSTER')[cluster_cols].mean().round(2)
print('Mean stats per cluster:')
print(cluster_profiles)